# Auditing the boundary model at the mouth

The interior-tide estimator sits on top of an ocean tide model evaluated
at the mouth. If that boundary series carries a phase error of its own,
the estimator would read it as estuarine transfer distributed along the
channel. This gate plants exactly that failure — a phase shift and a
gain error on the boundary, with no interior transfer at all — and asks
the machinery to attribute it correctly. The correction model acts per
tidal constituent $k$: amplitude gain $(1+\gamma_k)$ and time lag
$\Delta t_k$, fitted on the mouth pixels only, and only for constituents
the alias table declares estimable — the rest stay pinned to the prior.
The demands: the phase should be recovered
from the mouth pixels, the gain should not be (the same affine
degeneracy as in notebook 01), and the fictitious transfer should
disappear once the phase is corrected.

**You are here: 02.** Everything downstream trusts the ocean model at the mouth; this gate checks what happens when that trust is misplaced.

```text
+- the evidence chain ------------------------------------------------+
|  datacube -> water masks -> per-pixel wet/dry series                |
|    01 what is estimable  ->  02 boundary audit  ->  03 operator vs  |
|    gauges  ->  04 elevations vs truth  ->  05 uncertainty and       |
|    hydraulic layers  ->  06 negatives kept  ->  07 coast census     |
+---------------------------------------------------------------------+
```

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# run from the repo root so the results/ paths resolve
here = Path.cwd()
while not (here / "pyintertidal").is_dir():
    if here.parent == here:
        raise FileNotFoundError("repo root not found above " + str(Path.cwd()))
    here = here.parent
os.chdir(here)

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

In [2]:
r = load("results/m3_gate_sim/result.json")

print("planted:", r["plantado"])
print("recovered:", {k: r["recuperado"][k] for k in ("dt_min", "adopted")})
print("leak without correction (min):",
      np.round(r["T_sin_corregir_fuga"]["tau"], 1))
print("residual with correction (min):",
      np.round(r["T_corregido"]["tau"], 1),
      f"-> RMS {r['residuales']['rms_tau_min']:.1f}")
print("gate:", r["puerta"])

planted: {'gamma_m2': 0.08, 'dt_m2_min': 12.0, 'nota': 'gamma se planta pero NO se corrige (teorema afin): la puerta prueba que no falsea fase'}
recovered: {'dt_min': 16.0, 'adopted': True}
leak without correction (min): [ 0.  15.6 11.4 13.8 20.7 20.3]
residual with correction (min): [0.  1.7 0.8 3.7 7.  7. ] -> RMS 4.8
gate: {'dt_ok': True, 'T_plano_ok': True, 'fuga_demostrada_ok': True, 'PASA': True}
